# TML Assignment 3 - Robustness
## resnet34 + FAT-MART + EMA + SGDR, FRIENDLIER curriculum (tau cap 2, 100 epochs)

**Improvement: push the CLEAN axis via a friendlier FAT curriculum.** AWP failed on
resnet34 because it trades clean for robust, but r34's whole edge (and the leaderboard
0.6277) is its CLEAN accuracy (0.7525), which transfers 1:1 to the server. So we push
clean, not robust.

Single change vs the 0.6277 recipe: **FAT_MAX_TAU = 3 -> 2**. FAT freezes each sample's
perturbation tau steps after it is first misclassified; a smaller tau = friendlier (less
adversarial) examples = higher clean accuracy. tau now ramps 0->2 (cap at epoch 40)
instead of 0->3, keeping the back half friendlier. Expect clean up, PGD-20 slightly
down; net is + if the clean gain exceeds half the robust loss (clean transfers fully).

Everything else identical to the 0.6277 winner: resnet34, MART beta=3, FAT early-stopped
PGD (max 10 steps), EMA0.999, SGDR [40,60,80,100], crop+flip, 48k/2k, best-ckpt by
val_score, fp32 +-30 clamp. Stem `_r34_fatmart_tau2`.

Select by local CE-PGD-20 (code/eval_rank.py; server ~= local). Submit only if it beats
0.6304 local (0.6277 LB). Run cells top to bottom.

## 1. Setup

In [5]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
CUDA available: True
Device: Tesla T4


In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/tml_assignment3'
CKPT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoint dir:', CKPT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint dir: /content/drive/MyDrive/tml_assignment3/checkpoints


## 2. Download dataset

In [7]:
DATA_PATH = '/content/train.npz'
if not os.path.exists(DATA_PATH):
    !wget -q -O {DATA_PATH} "https://huggingface.co/datasets/SprintML/tml26_task3/resolve/main/train.npz"
print('Downloaded:', os.path.exists(DATA_PATH), os.path.getsize(DATA_PATH) / 1e6, 'MB')

Downloaded: True 126.604007 MB


## 3. Imports & hyperparameters

In [8]:
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision.models import resnet34
import torchvision.transforms.v2 as T
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True  # speeds up conv kernels for fixed input size

# ---- Hyperparameters (proven MART recipe, ported to resnet18 / 80 epochs) ----
NUM_CLASSES = 9
MODEL_NAME = 'resnet34'
BATCH_SIZE = 128
EPOCHS = 100
WARMUP_EPOCHS = 3
VAL_SIZE = 2000
SEED = 42

# Optimizer + LR schedule: warmup -> cosine, peak lr=0.1 (confirmed better than 0.02).
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4

# PGD attack (training) - CE-based, identical to all previous experiments (Madry 2018).
EPS = 8 / 255
ALPHA = 2 / 255
PGD_STEPS_TRAIN = 7

# PGD attack (evaluation - stronger)
PGD_STEPS_EVAL = 20

# FAT (Friendly Adversarial Training, Zhang et al. ICML 2020): early-stopped PGD.
# Inner attack stops perturbing each sample FAT_MAX_TAU steps after it is first
# misclassified -> "least-adversarial" (friendly) examples that preserve clean acc.
# tau ramps 0->4 over training (official schedule) so robustness is restored late.
FAT_NUM_STEPS = 10   # max inner PGD steps (most samples stop earlier)
FAT_TAU_STEP = 20    # tau increments by 1 every FAT_TAU_STEP epochs
FAT_MAX_TAU = 2      # friendlier curriculum (cap 2): leans toward CLEAN accuracy

# MART (Wang et al. 2020): loss = boosted-CE(adv) + MART_BETA * weighted KL(clean||adv).
# Targets misclassified examples explicitly. beta=3.0 was the project-best (vs 5/2).
MART_BETA = 3.0

# EMA of all floating-point params/buffers (incl. BatchNorm running stats), every step.
# Validation/checkpointing/submission all use ema_model.
EMA_DECAY = 0.999

# SGDR-3 (the EXACT schedule behind the 0.594 TRADES r18 model): 40-epoch initial
# cosine (lets clean fully converge), then two 20-epoch warm restarts at 60 and 80.
LR_CYCLE_BOUNDARIES = [40, 60, 80, 100]

# Mixed precision (PGD steps dominate per-batch cost)
USE_AMP = (device.type == 'cuda')

RESUME = True
CKPT_PATH = os.path.join(CKPT_DIR, f'{MODEL_NAME}_r34_fatmart_tau2.pt')
BEST_CKPT_PATH = os.path.join(CKPT_DIR, f'{MODEL_NAME}_r34_fatmart_tau2_best.pt')

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Device:', device, '| AMP:', USE_AMP)

Device: cuda | AMP: True


## 4. Data loading

In [9]:
data = np.load(DATA_PATH)
images = torch.from_numpy(data['images']).float() / 255.0  # (N, 3, 32, 32) in [0,1]
labels = torch.from_numpy(data['labels']).long()
print('Images:', images.shape, 'Labels:', labels.shape)

full_dataset = TensorDataset(images, labels)
train_size = len(full_dataset) - VAL_SIZE
train_set, val_set = random_split(
    full_dataset, [train_size, VAL_SIZE],
    generator=torch.Generator().manual_seed(SEED)
)
print('Train size:', len(train_set), 'Val size:', len(val_set))

# Standard CIFAR-style augmentation (applied to the clean image before PGD attack)
augment = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode='reflect'),
    T.RandomHorizontalFlip(),
])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False, num_workers=0)

Images: torch.Size([50000, 3, 32, 32]) Labels: torch.Size([50000])
Train size: 48000 Val size: 2000


## 5. Model

In [10]:
def build_model():
    model = resnet34(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model

model = build_model().to(device)

ema_model = build_model().to(device)
ema_model.load_state_dict(model.state_dict())
for p in ema_model.parameters():
    p.requires_grad_(False)
ema_model.eval()

# sanity check matching task_template.py
model.eval()
with torch.no_grad():
    out = model(torch.randn(1, 3, 32, 32, device=device))
assert out.shape == (1, NUM_CLASSES), out.shape
print('Output shape OK:', out.shape)

Output shape OK: torch.Size([1, 9])


## 6. PGD attack (identical to all previous PGD-AT experiments)

In [11]:
def fat_pgd_attack(model, x, y, eps, alpha, num_steps, tau, random_start=True):
    """FAT early-stopped PGD (Zhang et al. 2020). Per-sample, freeze the perturbation
    `tau` steps after the example is first misclassified (the "friendly"/least-
    adversarial example). tau=num_steps recovers standard PGD. Faithful vectorised
    version of the official earlystop() (fixed batch, no reorder)."""
    was_training = model.training
    model.eval()
    x_orig = x.detach()
    if random_start:
        x_adv = torch.clamp(x_orig + torch.empty_like(x_orig).uniform_(-eps, eps), 0., 1.).detach()
    else:
        x_adv = x_orig.clone().detach()
    control = torch.full((x.size(0),), float(tau), device=x.device)  # extra steps allowed post-misclassification
    frozen = torch.zeros(x.size(0), dtype=torch.bool, device=x.device)
    x_final = x_adv.clone().detach()
    for _ in range(num_steps):
        x_adv = x_adv.detach().requires_grad_(True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x_adv)
            loss = F.cross_entropy(logits, y)
        grad = torch.autograd.grad(loss, x_adv)[0]
        with torch.no_grad():
            mis = logits.argmax(1) != y
            stop_now = mis & (control <= 0) & (~frozen)   # misclassified for tau steps -> freeze
            x_final[stop_now] = x_adv[stop_now].detach()
            frozen = frozen | stop_now
            control = torch.where(mis & (~frozen), control - 1, control)
            active = (~frozen).float().view(-1, 1, 1, 1)
            x_adv = x_adv + alpha * grad.sign() * active   # update only un-frozen samples
            x_adv = torch.min(torch.max(x_adv, x_orig - eps), x_orig + eps)
            x_adv = torch.clamp(x_adv, 0., 1.)
    x_final[~frozen] = x_adv[~frozen].detach()             # never-stopped -> full attack
    if was_training:
        model.train()
    return x_final.detach()


def pgd_attack(model, x, y, eps, alpha, steps, random_start=True):
    """Untargeted L_inf PGD attack maximizing cross-entropy. x is in [0,1]. Model is
    set to eval() during attack generation so BatchNorm uses running statistics."""
    was_training = model.training
    model.eval()

    x_orig = x.detach()
    if random_start:
        delta = torch.empty_like(x_orig).uniform_(-eps, eps)
        x_adv = torch.clamp(x_orig + delta, 0.0, 1.0).detach()
    else:
        x_adv = x_orig.clone()

    for _ in range(steps):
        x_adv.requires_grad_(True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x_adv)
            loss = F.cross_entropy(logits, y)
        grad = torch.autograd.grad(loss, x_adv)[0]
        with torch.no_grad():
            x_adv = x_adv + alpha * grad.sign()
            x_adv = torch.clamp(x_adv, x_orig - eps, x_orig + eps)
            x_adv = torch.clamp(x_adv, 0.0, 1.0)

    if was_training:
        model.train()
    return x_adv.detach()


def fgsm_attack(model, x, y, eps):
    was_training = model.training
    model.eval()
    x_orig = x.detach().requires_grad_(True)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
        logits = model(x_orig)
        loss = F.cross_entropy(logits, y)
    grad = torch.autograd.grad(loss, x_orig)[0]
    x_adv = torch.clamp(x_orig + eps * grad.sign(), 0.0, 1.0).detach()
    if was_training:
        model.train()
    return x_adv

## 7. Optimizer, LR schedule (warmup + SGDR cosine), training loop

In [12]:
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)


def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    # SGDR-style cosine restarts. With LR_CYCLE_BOUNDARIES=[40,60,80]: a 40-epoch
    # cosine, then jump back up for a 20-epoch cosine (40-59), then another
    # 20-epoch cosine (60-79). The trailing default avoids StopIteration on the
    # final scheduler.step() at epoch == last boundary.
    total = next((b for b in LR_CYCLE_BOUNDARIES if epoch < b), LR_CYCLE_BOUNDARIES[-1])
    progress = (epoch - WARMUP_EPOCHS) / max(1, total - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

start_epoch = 0
best_score = -1.0
if RESUME and os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    ema_model.load_state_dict(ckpt['ema_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    print(f'Resumed from epoch {start_epoch}')
else:
    print('Starting fresh training')

if RESUME and os.path.exists(BEST_CKPT_PATH):
    best_score = torch.load(BEST_CKPT_PATH, map_location='cpu').get('score', -1.0)
    print(f'Loaded best score so far: {best_score:.4f}')

Starting fresh training


In [13]:
@torch.no_grad()
def evaluate_clean(model, loader):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total


def evaluate_pgd(model, loader, eps, alpha, steps, max_batches=None):
    model.eval()
    correct, total = 0, 0
    for i, (x, y) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        x_adv = pgd_attack(model, x, y, eps, alpha, steps)
        with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            pred = model(x_adv).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total

In [14]:
@torch.no_grad()
def update_ema(ema_model, model, decay):
    msd = model.state_dict()
    for k, v in ema_model.state_dict().items():
        if v.dtype.is_floating_point:
            v.mul_(decay).add_(msd[k], alpha=1 - decay)
        else:
            v.copy_(msd[k])


for epoch in range(start_epoch, EPOCHS):
    model.train()
    t0 = time.time()
    running_loss, running_correct, running_total = 0.0, 0, 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False)
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        x = augment(x)

        # Adversarial examples via the proven CE-based PGD-7 attack (no KL inside the loop)
        tau = min(FAT_MAX_TAU, epoch // FAT_TAU_STEP)
        x_adv = fat_pgd_attack(model, x, y, EPS, ALPHA, FAT_NUM_STEPS, tau)

        model.train()
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits_clean = model(x)
            logits_adv = model(x_adv)

        # MART loss (Wang et al. 2020), computed in fp32. Logits clamped to [-30,30]
        # to keep softmax away from fp16-induced underflow (same nan guard as TRADES).
        lc = logits_clean.float().clamp(-30, 30)
        la = logits_adv.float().clamp(-30, 30)
        adv_probs = F.softmax(la, dim=1)
        # Boosted CE: standard CE + margin term pushing down the most-likely WRONG
        # class on the adversarial example.
        tmp1 = torch.argsort(adv_probs, dim=1)[:, -2:]
        new_y = torch.where(tmp1[:, -1] == y, tmp1[:, -2], tmp1[:, -1])
        loss_adv = F.cross_entropy(la, y) + F.nll_loss(torch.log(1.0001 - adv_probs + 1e-12), new_y)
        # Robust KL(clean || adv), weighted per-sample by (1 - p_clean[y]).
        nat_probs = F.softmax(lc, dim=1)
        true_probs = nat_probs.gather(1, y.unsqueeze(1)).squeeze(1)
        kl_per = (nat_probs * (torch.log(nat_probs + 1e-12) - torch.log(adv_probs + 1e-12))).sum(dim=1)
        loss_robust = (kl_per * (1.0000001 - true_probs)).mean()

        loss = loss_adv + MART_BETA * loss_robust

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        update_ema(ema_model, model, EMA_DECAY)

        running_loss += loss.item() * y.size(0)
        running_correct += (logits_adv.argmax(dim=1) == y).sum().item()
        running_total += y.size(0)

        pbar.set_postfix(loss=running_loss / running_total, adv_acc=running_correct / running_total)

    scheduler.step()

    train_loss = running_loss / running_total
    train_adv_acc = running_correct / running_total
    # validation/checkpointing uses the EMA weights (what gets submitted)
    val_clean_acc = evaluate_clean(ema_model, val_loader)
    # cheap PGD-7 check on a few val batches each epoch (full PGD-20 eval done at the end)
    val_pgd_acc = evaluate_pgd(ema_model, val_loader, EPS, ALPHA, PGD_STEPS_TRAIN, max_batches=5)
    val_score = 0.5 * val_clean_acc + 0.5 * val_pgd_acc

    dt = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS} | lr {scheduler.get_last_lr()[0]:.4f} | '
          f'train_loss {train_loss:.4f} | train_adv_acc {train_adv_acc:.4f} | '
          f'val_clean_acc(ema) {val_clean_acc:.4f} | val_pgd7_acc(ema) {val_pgd_acc:.4f} | '
          f'val_score(ema) {val_score:.4f} | tau {tau} | {dt:.1f}s')

    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'ema_state_dict': ema_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'val_clean_acc': val_clean_acc,
        'val_pgd7_acc': val_pgd_acc,
        'score': val_score,
    }
    torch.save(ckpt, CKPT_PATH)

    if val_score > best_score:
        best_score = val_score
        torch.save(ckpt, BEST_CKPT_PATH)
        print(f'  -> new best (val_score={val_score:.4f}), saved to {BEST_CKPT_PATH}')

Epoch 1/100:   0%|          | 0/375 [00:00<?, ?it/s]

Epoch 1/100 | lr 0.0667 | train_loss 2.1743 | train_adv_acc 0.3057 | val_clean_acc(ema) 0.1160 | val_pgd7_acc(ema) 0.1219 | val_score(ema) 0.1189 | tau 0 | 83.9s
  -> new best (val_score=0.1189), saved to /content/drive/MyDrive/tml_assignment3/checkpoints/resnet34_r34_fatmart_tau2_best.pt


Epoch 2/100:   0%|          | 0/375 [00:00<?, ?it/s]

Epoch 2/100 | lr 0.1000 | train_loss 1.9073 | train_adv_acc 0.3704 | val_clean_acc(ema) 0.1545 | val_pgd7_acc(ema) 0.1227 | val_score(ema) 0.1386 | tau 0 | 74.0s
  -> new best (val_score=0.1386), saved to /content/drive/MyDrive/tml_assignment3/checkpoints/resnet34_r34_fatmart_tau2_best.pt


Epoch 3/100:   0%|          | 0/375 [00:00<?, ?it/s]

...

Epoch 98/100 | lr 0.0001 | train_loss 1.4918 | train_adv_acc 0.5492 | val_clean_acc(ema) 0.7625 | val_pgd7_acc(ema) 0.5312 | val_score(ema) 0.6469 | tau 2 | 77.6s


Epoch 99/100:   0%|          | 0/375 [00:00<?, ?it/s]

Epoch 99/100 | lr 0.0000 | train_loss 1.4910 | train_adv_acc 0.5503 | val_clean_acc(ema) 0.7635 | val_pgd7_acc(ema) 0.5234 | val_score(ema) 0.6435 | tau 2 | 78.5s


Epoch 100/100:   0%|          | 0/375 [00:00<?, ?it/s]

Epoch 100/100 | lr 0.0000 | train_loss 1.4906 | train_adv_acc 0.5490 | val_clean_acc(ema) 0.7630 | val_pgd7_acc(ema) 0.5273 | val_score(ema) 0.6452 | tau 2 | 78.0s


## 8. Final evaluation (best checkpoint, stronger PGD-20)

In [15]:
# Load the best checkpoint by val_score (not necessarily the last epoch)
best_ckpt = torch.load(BEST_CKPT_PATH, map_location=device)
model.load_state_dict(best_ckpt['ema_state_dict'])
print(f"Loaded best checkpoint (EMA weights) from epoch {best_ckpt['epoch']+1} (val_score={best_ckpt['score']:.4f})")

final_clean_acc = evaluate_clean(model, val_loader)
final_pgd20_acc = evaluate_pgd(model, val_loader, EPS, ALPHA, PGD_STEPS_EVAL)

model.eval()
fgsm_correct, fgsm_total = 0, 0
for x, y in val_loader:
    x, y = x.to(device), y.to(device)
    x_adv = fgsm_attack(model, x, y, EPS)
    with torch.no_grad():
        pred = model(x_adv).argmax(dim=1)
    fgsm_correct += (pred == y).sum().item()
    fgsm_total += y.size(0)
final_fgsm_acc = fgsm_correct / fgsm_total

print(f'Final clean accuracy:  {final_clean_acc:.4f}')
print(f'Final FGSM accuracy:   {final_fgsm_acc:.4f} (eps={EPS:.4f})')
print(f'Final PGD-20 accuracy: {final_pgd20_acc:.4f} (eps={EPS:.4f}, alpha={ALPHA:.4f})')
print(f'Estimated score (0.5*clean + 0.5*pgd20): {0.5*final_clean_acc + 0.5*final_pgd20_acc:.4f}')

Loaded best checkpoint (EMA weights) from epoch 96 (val_score=0.6470)
Final clean accuracy:  0.7620
Final FGSM accuracy:   0.5490 (eps=0.0314)
Final PGD-20 accuracy: 0.5100 (eps=0.0314, alpha=0.0078)
Estimated score (0.5*clean + 0.5*pgd20): 0.6360


## 9. Save submission state dict

In [16]:
SUBMIT_PATH = os.path.join(PROJECT_DIR, f'{MODEL_NAME}_r34_fatmart_tau2_submission.pt')

# sanity checks matching task_template.py / submission.py requirements
model.eval()
with torch.no_grad():
    out = model(torch.randn(1, 3, 32, 32, device=device))
assert out.shape == (1, NUM_CLASSES)

torch.save(model.state_dict(), SUBMIT_PATH)
print('Saved submission state dict to:', SUBMIT_PATH)
print('Model name for submission.py:', MODEL_NAME)

Saved submission state dict to: /content/drive/MyDrive/tml_assignment3/resnet34_r34_fatmart_tau2_submission.pt
Model name for submission.py: resnet34
